# UIT DSC 2026 LegalIR - Train Notebook (Cach 2)
Leader chi chay file nay. Khong paste code - chi import tu GitHub.
Repo: https://github.com/huynhnhatminh20/uit-dsc-2026-legal-raptor-KG (nop toan bo folder)


In [ ]:
# Cell 1: Clone GitHub + Cai lib (Leader chay dau tien)
import os, sys
!rm -rf uit-dsc-2026-legal-raptor-KG
!git clone https://github.com/huynhnhatminh20/uit-dsc-2026-legal-raptor-KG.git
sys.path.append('uit-dsc-2026-legal-raptor-KG')
sys.path.append('uit-dsc-2026-legal-raptor-KG/src')
print('Cloned:', os.listdir('uit-dsc-2026-legal-raptor-KG')[:10])
 # !pip install -q -r uit-dsc-2026-legal-raptor-KG/requirements.txt
!nvidia-smi
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Cell 2: Import modules cua 3 thanh vien (khong paste code)
from member_a import build_raptor, build_vector_store  # A
from member_b import build_graph, build_bm25, hybrid_retrieve  # B
from member_c import LegalModelWrapper, EmbeddingWrapper, LegalReranker, evaluate_recall_precision  # C
print('All modules imported')


In [ ]:
# Cell 3: Train (co checkpoint tu dong)
legal = LegalModelWrapper().load()  # VLSP2025-LegalSML/qwen3-4b-legal-pretrain 4-bit
emb = EmbeddingWrapper().load()  # BAAI/bge-m3
build_raptor(legal, emb)  # A: luu A_checkpoint/raptor_tree.pkl moi 50 chunk
build_vector_store(emb)  # A: FAISS
build_graph(legal)  # B: NetworkX
build_bm25()  # B: rank-bm25
print('Train done - checkpoints in /kaggle/working/*_checkpoint/')


In [ ]:
# Cell 4: Inference + Rerank -> submission.json (Leader chay)
import json, pathlib
from tqdm.auto import tqdm
reranker = LegalReranker().load()  # BAAI/bge-reranker-v2-m3
queries = json.loads(pathlib.Path('data_legalir/public-official.json').read_text(encoding='utf-8'))
submission = {}
for qid, item in tqdm(queries.items()):
    q = item['question'] if isinstance(item, dict) else item
    candidates = hybrid_retrieve(q, top_k=50)  # B: RRF
    top5 = reranker.rerank(q, candidates, top_k=5)  # C: cat cung 5
    submission[str(qid)] = {'answer': top5}
pathlib.Path('/kaggle/working/submission.json').write_text(json.dumps(submission, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', len(submission), 'queries -> /kaggle/working/submission.json')
import zipfile
with zipfile.ZipFile('/kaggle/working/submission.zip','w', zipfile.ZIP_DEFLATED) as z:
    z.write('/kaggle/working/submission.json','submission.json')
print('Zipped')


In [ ]:
# Cell 5: Doc ket qua + Evaluate tren warmup
import json, pathlib
from member_c import evaluate_recall_precision
gt = json.loads(pathlib.Path('data_legalir/train.json').read_text(encoding='utf-8'))
pred = json.loads(pathlib.Path('/kaggle/working/submission.json').read_text(encoding='utf-8'))
# chi danh gia tren tap nho neu muon
sample = {k: gt[k] for k in list(gt.keys())[:100]}
res = evaluate_recall_precision(sample, pred)
print(f"Recall@5={res['recall']:.4f} Precision@5={res['precision']:.4f} violated={res['violated']}")
print(open('/kaggle/working/submission.json',encoding='utf-8').read()[:500])
